In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

# Vector reduction using global memory 

In [ ]:
%%cuda
#include <stdio.h>

__global__ void reduce_to_place(float *input, int N)
{
    int idx = threadIdx.x + blockDim.x * blockIdx.x;
    for(int stride = 1; stride < blockDim.x; stride *= 2)
    {
        __syncthreads();
        if(idx % (2*stride) == 0 && idx + stride < N)
        {
            input[idx] += input[idx + stride];
        }
    }
    if(threadIdx.x == 0)
    {
        input[blockIdx.x] = input[blockIdx.x * blockDim.x];
    }
}

float cpu_reduce(float *input, int n)
{
    float sum = 0.0;
    for(int i = 0; i < n; i++)
    {
        sum += input[i];
    }
    return sum;
}

int main()
{
    int N = 16;
    float h_input[16] = {10,1,8,-1,0,-2,3,4,-2,-3, 2,7,0,11,0,2};

    printf("CPU reduction result: %f\n", cpu_reduce(h_input, N));
    
    float *d_input;
    cudaMalloc(&d_input, N*sizeof(float));
    cudaMemcpy(d_input, h_input, N*sizeof(float), cudaMemcpyHostToDevice);
    reduce_to_place<<<2, 8>>>(d_input, N);
    cudaDeviceSynchronize();
    cudaMemcpy(h_input, d_input, N*sizeof(float), cudaMemcpyDeviceToHost);
    printf("Reduction result: %f\n", h_input[0]);
    reduce_to_place<<<1, 2>>>(d_input, 2);
    cudaMemcpy(h_input, d_input, N*sizeof(float), cudaMemcpyDeviceToHost);
    printf("Final reduction result: %f\n", h_input[0]);
    cudaFree(d_input);
}

In [ ]:
%%cuda
#include <stdio.h>

__global__ void reduce_to_place(float *input, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    for(int stride = 1; stride < blockDim.x; stride *= 2)
    {
        __syncthreads();
        /*if(idx % (2*stride) == 0 && idx + stride < n)
        {
            input[idx] += input[idx + stride];
        }*/
        //this is efficient as modulas operation is expensive as compared to the above commented condition
        if(idx + stride < n)
        {
            input[idx] += input[idx + stride];
        }
    }

    __syncthreads();
    
    if(threadIdx.x == 0)
    {
        input[blockIdx.x] = input[blockIdx.x * blockDim.x];
    }
}

float cpu_reduce(float *input, int n)
{
    float sum = 0.0;
    for(int i = 0; i < n; i++)
    {
        sum += input[i];
    }
    return sum;
}

int main()
{
    int N = 1024*1024;
    float *h_input = (float*)malloc(N*sizeof(float));
    for(int i = 0; i < N; i++)
    {
        h_input[i] = i * 1.0f;
    }
    clock_t start = clock();
    float cpu_result = cpu_reduce(h_input, N);
    clock_t end = clock();
    double cpu_time = (double(end - start)) * 1000.0 / CLOCKS_PER_SEC;
    printf("CPU time: %f milliseconds\n", cpu_time);
    printf("CPU reduction result: %f\n", cpu_reduce(h_input, N));

    float *d_input;
    cudaMalloc(&d_input, N*sizeof(float));
    cudaMemcpy(d_input, h_input, N*sizeof(float), cudaMemcpyHostToDevice);
    
    cudaEvent_t gpu_start, gpu_end;
    cudaEventCreate(&gpu_start);
    cudaEventCreate(&gpu_end);
    cudaEventRecord(gpu_start);
    
    int blocksize = 256;
    int gridsize = (N+ blocksize - 1)/blocksize;
    int current_size = N;
    while(gridsize > 1)
    {
        reduce_to_place<<<gridsize, blocksize>>>(d_input, current_size);
        cudaDeviceSynchronize();
        current_size = gridsize;
        gridsize = (gridsize + blocksize - 1) / blocksize;
    }
    
    reduce_to_place<<<1, blocksize>>>(d_input, current_size);
    cudaDeviceSynchronize();
    
    cudaEventRecord(gpu_end);
    cudaEventSynchronize(gpu_end);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, gpu_start, gpu_end);
    
    cudaMemcpy(h_input, d_input, sizeof(float), cudaMemcpyDeviceToHost);
    printf("GPU reduction result: %f\n", h_input[0]);
    printf("GPU time: %f milliseconds\n", gpu_time);
    printf("Speedup: %f x\n", cpu_time / gpu_time);
    free(h_input);
    cudaFree(d_input);
    cudaEventDestroy(gpu_start);
    cudaEventDestroy(gpu_end);
}

# Optimizing and assigning more work per thread
In the below code we are going to:
- add the next 256 elements to the first 256 elements
- now you can execute the same previous application.
- because block size = number of elements = 256 as previous.
- that means that you need to add the first step to the code.
- Moreover, you need half of the gridsize;

In [ ]:
%%cuda
#include <stdio.h>

__global__ void reduce_to_place(float *input, int N)
{
    int idx = threadIdx.x + (2 * blockDim.x * blockIdx.x);

    if(idx + blockDim.x < N)
    {
        input[idx] += input[idx + blockDim.x];
    }

    __syncthreads();

    for(int stride = blockDim.x >> 1; stride > 0; stride >>= 1)
    {
        if(idx + stride < N)
        {
            input[idx] += input[idx + stride];
        }
        __syncthreads();
    }

    if(threadIdx.x == 0)
    {
        input[blockIdx.x] = input[idx];
    }
}

float cpu_reduce(float *input, int n)
{
    float sum = 0.0;
    for(int i = 0; i < n; i++)
    {
        sum += input[i];
    }
    return sum;
}

int main()
{
    int N = 1024*1024;
    float *h_input = (float*)malloc(N*sizeof(float));
    for(int i = 0; i < N; i++)
    {
        h_input[i] = i * 1.0f;
    }
    clock_t start = clock();
    float cpu_result = cpu_reduce(h_input, N);
    clock_t end = clock();
    double cpu_time = (double(end - start)) * 1000.0 / CLOCKS_PER_SEC;
    printf("CPU time: %f milliseconds\n", cpu_time);
    printf("CPU reduction result: %f\n", cpu_reduce(h_input, N));

    float *d_input;
    cudaMalloc(&d_input, N*sizeof(float));
    cudaMemcpy(d_input, h_input, N*sizeof(float), cudaMemcpyHostToDevice);
    
    cudaEvent_t gpu_start, gpu_end;
    cudaEventCreate(&gpu_start);
    cudaEventCreate(&gpu_end);
    cudaEventRecord(gpu_start);
    
    int blocksize = 256;
    int gridsize = (N + (2 * blocksize) - 1) / (2 * blocksize);
    int current_size = N;
    while(gridsize > 1)
    {
        reduce_to_place<<<gridsize, blocksize>>>(d_input, current_size);
        cudaDeviceSynchronize();
        current_size = gridsize;
        gridsize = (gridsize + (2 * blocksize) - 1) / (2 * blocksize);
    }
    
    reduce_to_place<<<1, blocksize>>>(d_input, current_size);
    cudaDeviceSynchronize();
    
    cudaEventRecord(gpu_end);
    cudaEventSynchronize(gpu_end);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, gpu_start, gpu_end);
    
    cudaMemcpy(h_input, d_input, sizeof(float), cudaMemcpyDeviceToHost);
    printf("GPU reduction result: %f\n", h_input[0]);
    printf("GPU time: %f milliseconds\n", gpu_time);
    printf("Speedup: %f x\n", cpu_time / gpu_time);
    free(h_input);
    cudaFree(d_input);
    cudaEventDestroy(gpu_start);
    cudaEventDestroy(gpu_end);
}

# Vector Reduction and optimization using shared memory

In [ ]:
%%cuda
#include <stdio.h>

__global__ void reduce_to_place(float *input, int N)
{
    __shared__ float shared_data[256];
    int idx = threadIdx.x + (2 * blockDim.x * blockIdx.x);

    if(idx + blockDim.x < N)
    {
        shared_data[threadIdx.x] = input[idx] + input[idx + blockDim.x];
    }
    else if(idx < N)
    {
        shared_data[threadIdx.x] = input[idx];
    }
    else
    {
        shared_data[threadIdx.x] = 0.0f;
    }

    __syncthreads();

    for(int stride = blockDim.x >> 1; stride > 0; stride >>= 1)
    {
        if(threadIdx.x < stride)
        {
            shared_data[threadIdx.x] += shared_data[threadIdx.x + stride];
        }
        __syncthreads();
    }

    if(threadIdx.x == 0)
    {
        input[blockIdx.x] = shared_data[0];
    }
}

float cpu_reduce(float *input, int n)
{
    float sum = 0.0;
    for(int i = 0; i < n; i++)
    {
        sum += input[i];
    }
    return sum;
}

int main()
{
    int N = 1024*1024;
    float *h_input = (float*)malloc(N*sizeof(float));
    for(int i = 0; i < N; i++)
    {
        h_input[i] = i * 1.0f;
    }
    clock_t start = clock();
    float cpu_result = cpu_reduce(h_input, N);
    clock_t end = clock();
    double cpu_time = (double(end - start)) * 1000.0 / CLOCKS_PER_SEC;
    printf("CPU time: %f milliseconds\n", cpu_time);
    printf("CPU reduction result: %f\n", cpu_reduce(h_input, N));

    float *d_input;
    cudaMalloc(&d_input, N*sizeof(float));
    cudaMemcpy(d_input, h_input, N*sizeof(float), cudaMemcpyHostToDevice);
    
    cudaEvent_t gpu_start, gpu_end;
    cudaEventCreate(&gpu_start);
    cudaEventCreate(&gpu_end);
    cudaEventRecord(gpu_start);
    
    int blocksize = 256;
    int gridsize = (N + (2 * blocksize) - 1) / (2 * blocksize);
    int current_size = N;
    while(gridsize > 1)
    {
        reduce_to_place<<<gridsize, blocksize>>>(d_input, current_size);
        cudaDeviceSynchronize();
        current_size = gridsize;
        gridsize = (gridsize + (2 * blocksize) - 1) / (2 * blocksize);
    }
    
    reduce_to_place<<<1, blocksize>>>(d_input, current_size);
    cudaDeviceSynchronize();
    
    cudaEventRecord(gpu_end);
    cudaEventSynchronize(gpu_end);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, gpu_start, gpu_end);
    
    cudaMemcpy(h_input, d_input, sizeof(float), cudaMemcpyDeviceToHost);
    printf("GPU reduction result: %f\n", h_input[0]);
    printf("GPU time: %f milliseconds\n", gpu_time);
    printf("Speedup: %f x\n", cpu_time / gpu_time);
    free(h_input);
    cudaFree(d_input);
    cudaEventDestroy(gpu_start);
    cudaEventDestroy(gpu_end);
}